# T6 - Image Reconstruction (standalone)
Input: beamformed RF `(depth x lines)` from T5. Output: conditioned RF for T7 (B-mode). Steps: DC removal per scan line, zero-phase band-pass around the centre frequency, envelope for inspection.

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
import receiver_beamforming as rb
from image_reconstruction import image_reconstruction, ReconConfig

## Input
Synthetic channel data with a DC offset and noise is beamformed with T5. Replace `rf` with your own `(depth x lines)` array and `z_axis` if you have one.

In [ ]:
c, fs, f0, n = 1540.0, 40e6, 5e6, 128
ex = (np.arange(n) - (n - 1) / 2) * c / f0 / 2
t = np.arange(-1.2e-6, 50e-6, 1 / fs)
rx = np.zeros((n, t.size))
for x, z in [(-3e-3, 12e-3), (0.0, 22e-3), (3e-3, 32e-3)]:
    tau = z / c + np.hypot(x - ex, z) / c
    d = t[None, :] - tau[:, None]
    rx += np.sin(2*np.pi*f0*d) * np.exp(-0.5 * (d / (3/(2.5*f0)))**2)
print("synthetic channel data:", rx.shape)
rx = rx + 0.4 * np.abs(rx).max() * np.random.default_rng(1).normal(size=rx.shape) + 0.05
bf = rb.beamform(rb.make_receiver_input(rx, t, ex, fs_hz=fs, c0=c, f0_hz=f0), rb.RxConfig())
rf, z_axis, x_lines = bf.beamformed_rf, bf.z_axis, bf.x_lines
print('beamformed_rf', rf.shape)

## Reconstruct

In [ ]:
rec = image_reconstruction(rf, z_axis, c=c, f0_hz=f0, config=ReconConfig())
print(rec.rf.shape, '|', '; '.join(rec.notes), f'| fs along depth {rec.fs_depth_hz/1e6:.1f} MHz')
li = rf.shape[1]//2; zi = int(np.argmax(np.abs(rf[:, li]))); sl = slice(max(zi-150, 0), zi+150)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(z_axis[sl]*1e3, rf[sl, li], lw=.8, label='before'); ax[0].plot(z_axis[sl]*1e3, rec.rf[sl, li], lw=.8, label='after'); ax[0].plot(z_axis[sl]*1e3, rec.envelope[sl, li], 'k', lw=1.5, label='envelope'); ax[0].legend(); ax[0].set(title='centre line', xlabel='depth (mm)')
f = np.fft.rfftfreq(rf.shape[0], 1/rec.fs_depth_hz)/1e6
ax[1].plot(f, np.abs(np.fft.rfft(rf[:, li])), lw=.8, label='before'); ax[1].plot(f, np.abs(np.fft.rfft(rec.rf[:, li])), lw=.8, label='after'); ax[1].set(title='spectrum', xlabel='MHz', xlim=(0, 15)); ax[1].legend()
ext = [x_lines[0]*1e3, x_lines[-1]*1e3, z_axis[-1]*1e3, z_axis[0]*1e3]
ax[2].imshow(rec.envelope, aspect='auto', cmap='gray', extent=ext); ax[2].set(title='envelope image', xlabel='x (mm)', ylabel='z (mm)')
plt.tight_layout(); plt.show()

## Switch it off and compare
`ReconConfig(enabled=False)` returns the input unchanged.

In [ ]:
off = image_reconstruction(rf, z_axis, c=c, f0_hz=f0, config=ReconConfig(enabled=False))
print('identical when disabled:', np.allclose(off.rf, rf))